# 14 — Phiên 60 phút: chạy test một lần và so sánh với 30 phút

Người dùng duyệt ngày 24/11/2026 một **ngoại lệ có ghi nhận** với quy tắc "test chỉ chạy một lần": đọc test thêm một lần cho biến thể phiên 60 phút đã tune ở notebook 13. Kết quả 30 phút ở `reports/final/` **vẫn là số chính thức**; notebook này là báo cáo phụ và không được dùng để chọn lại giao thức chính.

- Trước khi đọc test: ghi mã băm của báo cáo validation 60 phút, mô hình `models/session60/` và dữ liệu `data/processed/session60/` vào `frozen.json`.
- Hai biến thể như notebook 10 (D2): **train_only** (mô hình đã tune, fit trên train 60 phút) và **train_validation_refit** (biến thể chính: fit lại trên train + validation 60 phút với đúng cấu hình; GRU4Rec/SR-GNN dùng số epoch đã chọn; vocabulary, popularity, láng giềng, luật và `known_at` tính lại). Reranker giữ mô hình học trên validation 60 phút, chỉ thay ứng viên/đặc trưng theo biến thể.
- Hai bộ query đánh giá:
  - **`q30`** — đúng bộ query test chính thức (phiên 30 phút): so sánh trực tiếp với `reports/final`; paired bootstrap với Top-K 30 phút đã lưu (VSTAN, GRU4Rec, train_only).
  - **`q60`** — bộ query test theo phiên 60 phút: kết quả của giao thức 60 phút trọn vẹn (bộ query khác, chỉ đọc như mô tả).
- **Diễn tập:** đặt `RR_SESSION60_SPLIT=validation` để chạy toàn bộ đường code trên validation (refit chỉ fit train với 1 epoch, ghi vào `reports/session60/rehearsal/`). Lần chạy thật dùng mặc định `test` và chỉ chạy **một lần**.

## 1. Chế độ chạy, đóng băng cấu hình

In [ ]:
from pathlib import Path
import os, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, save_verified, utc_now, verify_inputs, write_json
from rr.ablation import score_models, topk_matrix
from rr.candidates import CandidateFilter
from rr.evaluation import run
from rr.metrics import bootstrap_ci, evaluate, paired_bootstrap
from rr.models import popularity
from rr.samples import build_vocab, prefix_events
from rr.rerank import add_features, candidate_table, ranked_lists

SPLIT = os.environ.get("RR_SESSION60_SPLIT", "test")
require(SPLIT in {"test", "validation"}, "RR_SESSION60_SPLIT phải là test hoặc validation.")
REHEARSAL = SPLIT == "validation"
OUT = ROOT / ("reports/session60/rehearsal" if REHEARSAL else "reports/session60/test")
PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
DATA60 = PROCESSED / "session60"
MODEL60 = ROOT / "models/session60"
SEED = PROTOCOL["seed"]
BOOT, PAIRED = PROTOCOL["evaluation"]["bootstrap"], PROTOCOL["evaluation"]["paired_bootstrap"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
WIDTH = 200
MODELS = ["popularity", "spop", "repeat_last", "itemknn", "sknn", "vsknn", "stan", "vstan", "sr", "gru4rec", "srgnn", "reranker"]
torch.set_num_threads(4)

LOG = os.environ.get("RR_PROGRESS_LOG")
def log(msg):
    line = f"{pd.Timestamp.now(tz='UTC'):%H:%M:%S} [14 {SPLIT}] {msg}"
    print(line, flush=True)
    if LOG:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")

if not REHEARSAL:
    require(not (OUT / "metrics.json").exists(), "Test 60 phút đã được chạy; không chạy lại (ngoại lệ được duyệt chỉ cho một lần).")
    require((ROOT / "reports/session60/rehearsal/metrics.json").exists(), "Chạy diễn tập RR_SESSION60_SPLIT=validation trước.")
val60 = read_json(ROOT / "reports/session60/validation.json")
FROZEN_FILES = ["configs/protocol.json", "reports/session60/validation.json", "reports/final/metrics.json",
                *[f"models/session60/{d}" for d in ["gru4rec/model.pt", "gru4rec/config.json", "srgnn/model.pt", "srgnn/config.json",
                                                   "reranker/model.txt", "reranker/config.json"]]]
frozen = {p: fingerprint(ROOT / p) for p in FROZEN_FILES}
for p in FROZEN_FILES:
    if p in val60["artifacts"]:
        require(frozen[p] == val60["artifacts"][p], f"{p} khác lúc notebook 13 lưu.")
require(val60["protocol"]["sha256"] == frozen["configs/protocol.json"]["sha256"], "protocol.json đổi sau notebook 13.")
OUT.mkdir(parents=True, exist_ok=True)
write_json(OUT / "frozen.json", {"generated_at_utc": utc_now(), "split": SPLIT, "files": frozen})
log("đã đóng băng cấu hình")

## 2. Kiểm tra đầu vào và ví dụ biên

Đầu vào 60 phút khớp SHA-256 trong `reports/session60/validation.json`; đầu vào 30 phút khớp report của notebook 02/03/04 và Top-K đã lưu của notebook 10.

In [ ]:
prev = {n: read_json(ROOT / f"reports/{n}/metrics.json") for n in ["session_split", "metadata", "samples"]}
ref30 = read_json(ROOT / ("reports/final_rehearsal/metrics.json" if REHEARSAL else "reports/final/metrics.json"))
expected = {p: v for p, v in val60["artifacts"].items() if p.startswith("data/")}
expected["data/processed/events_sessionized.parquet"] = prev["session_split"]["artifacts"]["data/processed/events_sessionized.parquet"]
expected["data/processed/item_property_history.parquet"] = prev["metadata"]["artifacts"]["data/processed/item_property_history.parquet"]
expected.update({p: v for p, v in prev["samples"]["artifacts"].items() if f"/{SPLIT}_" in p})
expected[ref30["saved_topk"]["path"]] = ref30["saved_topk"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", len(inputs), "tệp")

# ví dụ biên: Top-K dài -> ma trận; paired bootstrap của hai danh sách giống nhau cho hiệu bằng 0
_m = topk_matrix(pd.DataFrame({"query_id": [3, 3], "rank": [1, 2], "itemid": [8, 9]}), [3, 4], 3)
assert _m.tolist() == [[8, 9, -1], [-1, -1, -1]]
assert paired_bootstrap([0.5, 0.2], [0.5, 0.2], 50, 0)["mean_diff"] == 0
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu chung

In [ ]:
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
known = pd.read_parquet(DATA60 / "item_known_at.parquet")
ev60 = pd.read_parquet(DATA60 / "events_sessionized.parquet",
                       columns=["session_id", "group_index", "itemid", "event", "timestamp", "split", "visitorid"])
ev30 = pd.read_parquet(PROCESSED / "events_sessionized.parquet", columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
tcols = ["query_id", "itemid", "target_is_repeat", "has_strong_behavior"]
EVAL = {"q60": (pd.read_parquet(DATA60 / f"samples/{SPLIT}_queries.parquet"), pd.read_parquet(DATA60 / f"samples/{SPLIT}_targets.parquet", columns=tcols),
                ev60[ev60["split"] == SPLIT]),
        "q30": (pd.read_parquet(PROCESSED / f"samples/{SPLIT}_queries.parquet"), pd.read_parquet(PROCESSED / f"samples/{SPLIT}_targets.parquet", columns=tcols),
                ev30[ev30["split"] == SPLIT])}
del ev30
for name, (q, t, e) in EVAL.items():
    print(f"{name}: {len(q):,} query, {len(t):,} lượt target")

gcfg, scfg = read_json(MODEL60 / "gru4rec/config.json"), read_json(MODEL60 / "srgnn/config.json")
rcfg = read_json(MODEL60 / "reranker/config.json")
booster = lgb.Booster(model_str=(MODEL60 / "reranker/model.txt").read_text(encoding="utf-8"))
CFG = {"popularity": val60["popularity"]["selected"], "spop_weights": val60["spop"]["runs"][val60["spop"]["best"]]["weights"],
       "knn_selected": val60["knn"]["selected"], "gru": gcfg, "srgnn": scfg, "fallback_top_n": FALLBACK_TOP_N, "max_groups": MAX_GROUPS,
       "max_len": PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1, "seed": SEED, "width": WIDTH, "root": ROOT, "model_dir": MODEL60}
VARIANTS = {"train_only": {"fit_splits": ["train"], "known_at": "known_at_train_ms", "neural": "load"},
            "train_validation_refit": {"fit_splits": ["train", "validation"], "known_at": "known_at_trainval_ms", "neural": "train"}}
if REHEARSAL:  # exercise the refit path without leakage: refit on train only, 1 epoch
    VARIANTS["train_validation_refit"] = {"fit_splits": ["train"], "known_at": "known_at_train_ms", "neural": "train", "epochs": 1}

## 4. Chạy từng biến thể trên hai bộ query

Mỗi biến thể: fit mọi mô hình trên dữ liệu fit 60 phút, chấm `q60` rồi dùng lại đúng các mô hình đó để chấm `q30`. Reranker xếp lại hợp ứng viên của kNN tốt nhất và GRU4Rec của chính biến thể.

In [ ]:
ALL, PER_QUERY, TIMES = {}, [], {}
for vname, vcfg in VARIANTS.items():
    t_var = time.perf_counter()
    fit_events = ev60[ev60["split"].isin(vcfg["fit_splits"])].reset_index(drop=True)
    cfg = dict(CFG)
    if "epochs" in vcfg:
        cfg["gru"], cfg["srgnn"] = {**gcfg, "epochs": vcfg["epochs"]}, {**scfg, "epochs": vcfg["epochs"]}
    fit_samples = None
    if vcfg["neural"] == "train":
        fq = [pd.read_parquet(DATA60 / f"samples/{s}_queries.parquet") for s in vcfg["fit_splits"]]
        ft = [pd.read_parquet(DATA60 / f"samples/{s}_targets.parquet", columns=tcols) for s in vcfg["fit_splits"]]
        fit_samples = (pd.concat(fq, ignore_index=True), pd.concat(ft, ignore_index=True))
    vocab = build_vocab(fit_events)
    if vname == "train_only":
        require(vocab.equals(pd.read_parquet(DATA60 / "item_vocab.parquet")), "Vocabulary train khác notebook 13.")
    elif not REHEARSAL:
        require(vocab.equals(pd.read_parquet(DATA60 / "item_vocab_trainval.parquet")), "Vocabulary train + validation khác notebook 13.")
    flt = CandidateFilter.from_frames(vocab, known, history, vcfg["known_at"])
    fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(fit_events, CFG["popularity"]["fallback_weights"]))
    neural_models, res_v = None, {}
    for ename, (eq, et, ee) in EVAL.items():
        log(f"{vname} / {ename}: bắt đầu")
        mode = vcfg["neural"] if neural_models is None else "reuse"
        res, neural_models = score_models(fit_events, ee, eq, et, known, history, vcfg["known_at"], cfg, mode,
                                          fit_samples=fit_samples, neural_models=neural_models, log=log,
                                          keep_lists=(rcfg["sources"]["knn"], "gru4rec"))
        t0 = time.perf_counter()
        qids = eq["query_id"].to_numpy()
        prefix = prefix_events(eq, ee, ("itemid", "event"), max_groups=MAX_GROUPS)
        src = {"knn": res[rcfg["sources"]["knn"]].pop("lists"), "gru": res["gru4rec"].pop("lists")}
        table = add_features(candidate_table(qids, src, prefix, top_n=rcfg["top_n"]), eq, prefix, fit_events, history)
        rep, pq, *_ = run(*ranked_lists(table, booster.predict(table[rcfg["features"]]), qids, WIDTH), eq, et, flt, fb_items, fb_scores)
        res["reranker"] = {"report": rep, "per_query": pq, "fit_seconds": 0.0, "query_seconds": round(time.perf_counter() - t0, 1)}
        del table, src, prefix
        for name in MODELS:
            r = res[name]
            nd = r["per_query"]["ndcg@20"].to_numpy()
            r["report"]["bootstrap_ndcg@20"] = bootstrap_ci(nd, BOOT["n_resamples"], BOOT["seed"], BOOT["level"])
            r["report"]["paired_vs_repeat_last"] = paired_bootstrap(nd, res["repeat_last"]["per_query"]["ndcg@20"].to_numpy(),
                                                                    PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])
            PER_QUERY.append(pd.DataFrame({"variant": vname, "eval_set": ename, "model": name, "query_id": r["per_query"]["query_id"].to_numpy(),
                                           "hit@20": r["per_query"]["hit@20"].to_numpy(), "ndcg@20": nd}))
        res_v[ename] = res
        log(f"{vname} / {ename}: xong | " + ", ".join(f"{n}={res[n]['report']['all']['ndcg@20']:.4f}" for n in ["repeat_last", "vstan", "gru4rec", "srgnn", "reranker"]))
    ALL[vname] = res_v
    TIMES[vname] = round(time.perf_counter() - t_var, 1)
    del neural_models, fit_events

## 5. So sánh với 30 phút

- Bảng chính: trên **cùng bộ query `q30`**, so cột 30 phút (số chính thức) với cột 60 phút, kèm khoảng tin cậy bootstrap 95% của từng bên.
- Paired bootstrap (cùng query, cùng lần lấy mẫu lại) cho VSTAN và GRU4Rec ở biến thể train_only, nơi có Top-K 30 phút đã lưu.

In [ ]:
saved30 = pd.read_parquet(ROOT / ref30["saved_topk"]["path"])
q30, t30, _ = EVAL["q30"]
PAIRED30 = {}
for name in sorted(saved30["model"].unique()):
    rep30, pq30 = evaluate(topk_matrix(saved30[saved30["model"] == name], q30["query_id"].to_numpy(), 20), q30, t30)
    require(abs(rep30["all"]["ndcg@20"] - ref30["results"]["train_only"][name]["main"]["all"]["ndcg@20"]) < 1e-12,
            f"Top-K 30 phút đã lưu của {name} không khớp báo cáo notebook 10.")
    pq60 = ALL["train_only"]["q30"][name]["per_query"]
    require((pq60["query_id"].to_numpy() == pq30["query_id"].to_numpy()).all(), "Hai bên phải cùng thứ tự query.")
    PAIRED30[name] = paired_bootstrap(pq60["ndcg@20"].to_numpy(), pq30["ndcg@20"].to_numpy(), PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])

def compare(vname, part="all"):
    rows = {}
    for n in MODELS:
        r30, r60 = ref30["results"][vname][n]["main"], ALL[vname]["q30"][n]["report"]
        rows[n] = {"30 phút": r30[part]["ndcg@20"], "60 phút (q30)": r60[part]["ndcg@20"], "Δ": r60[part]["ndcg@20"] - r30[part]["ndcg@20"],
                   "60 phút (q60)": ALL[vname]["q60"][n]["report"][part]["ndcg@20"]}
        if part == "all":
            rows[n].update({"CI 30": f"[{r30['bootstrap_ndcg@20']['low']:.4f}, {r30['bootstrap_ndcg@20']['high']:.4f}]",
                            "CI 60 (q30)": f"[{r60['bootstrap_ndcg@20']['low']:.4f}, {r60['bootstrap_ndcg@20']['high']:.4f}]"})
    return pd.DataFrame(rows).T

TABLES = {}
for vname in VARIANTS:
    for part, title in [("all", "Bài toán 1"), ("strong_behavior_targets", "Bài toán 2"), ("new_targets", "Target mới"), ("repeat_targets", "Target lặp")]:
        TABLES[(vname, part)] = compare(vname, part)
        print(f"{vname} — {title}")
        display(TABLES[(vname, part)])
display(pd.DataFrame(PAIRED30).T[["mean_diff", "low", "high", "p_not_better"]])

## 6. Lưu báo cáo

In [ ]:
def slim(res):
    return {n: {"report": r["report"], "fit_seconds": r["fit_seconds"], "query_seconds": r["query_seconds"]} for n, r in res.items()}

def md_table(frame, first="Mô hình"):
    cols = list(frame.columns)
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    return (f"| {first} | " + " | ".join(cols) + " |\n|" + " --- |" * (len(cols) + 1) + "\n"
            + "".join(f"| {i} | " + " | ".join(fmt(v) for v in row) + " |\n" for i, row in frame.iterrows()))

require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
require(all(fingerprint(ROOT / p) == v for p, v in frozen.items()), "Cấu hình đóng băng bị thay đổi trong lúc chạy.")
pq_info = save_verified(pd.concat(PER_QUERY, ignore_index=True), DATA60 / f"{SPLIT}_per_query{'_rehearsal' if REHEARSAL else ''}.parquet",
                        ["variant", "eval_set", "model", "query_id"], ROOT)
MAIN = "train_validation_refit"
write_json(OUT / "metrics.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__, "lightgbm_version": lgb.__version__,
    "notebook": "notebooks/14_phien_60_phut_test.ipynb", "split": SPLIT, "rehearsal": REHEARSAL,
    "status_note": "secondary report; official results stay reports/final (30-minute sessions)",
    "approved_exception": "user approved on 2026-11-24 one additional test read for the 60-minute variant",
    "protocol": {"version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)}, "frozen": frozen, "inputs": inputs,
    "n_queries": {e: int(len(q)) for e, (q, _, _) in EVAL.items()}, "primary_variant": MAIN, "variant_seconds": TIMES,
    "results": {v: {e: slim(r) for e, r in res.items()} for v, res in ALL.items()},
    "paired_60_vs_30_on_q30_train_only": PAIRED30, "per_query": pq_info, "status": "PASS"})

lines = [f"# Phiên 60 phút so với 30 phút trên {SPLIT}{' (DIỄN TẬP — không dùng để báo cáo)' if REHEARSAL else ''}\n\n",
         f"Sinh bởi `notebooks/14_phien_60_phut_test.ipynb` lúc {utc_now()}. Báo cáo phụ: số chính thức vẫn là `reports/final` (phiên 30 phút). "
         f"Cấu hình 60 phút được tune lại trên validation 60 phút (notebook 13). `q30`: {len(EVAL['q30'][0]):,} query của giao thức chính; "
         f"`q60`: {len(EVAL['q60'][0]):,} query theo phiên 60 phút. Số liệu: NDCG@20. Biến thể chính: `{MAIN}`.\n"]
for vname in VARIANTS:
    lines.append(f"\n## Biến thể `{vname}`\n")
    for part, title in [("all", "Bài toán 1 — tương tác kế tiếp"), ("strong_behavior_targets", "Bài toán 2 — thêm giỏ/mua kế tiếp"),
                        ("new_targets", "Chỉ target mới"), ("repeat_targets", "Chỉ target lặp")]:
        lines.append(f"\n### {title}\n\n" + md_table(TABLES[(vname, part)]))
lines.append("\n## Paired bootstrap 60 − 30 phút trên cùng query `q30` (train_only)\n\n" +
             md_table(pd.DataFrame(PAIRED30).T[["mean_diff", "low", "high", "p_not_better"]]))
(OUT / "BAO_CAO_PHIEN_60_PHUT.md").write_text("".join(lines), encoding="utf-8", newline="\n")
log("xong, báo cáo: " + rel(OUT / "metrics.json", ROOT))

## Kết luận

- **Bài toán 1, cùng 92.468 query test của giao thức chính:** phiên 60 phút gần như không đổi kết quả.
  - kNN theo phiên chỉ tăng khoảng +0,001.
  - Ở GRU4Rec, Δ là +0,0007 với `train_only` (paired bootstrap CI [−0,0001; +0,0016]) và −0,0001 với `train_validation_refit`.
  - SR-GNN tăng nhiều nhất, khoảng +0,003 đến +0,004.
  - Reranker (mô hình tốt nhất) lệch −0,0002 đến −0,0003, nằm trong khoảng CI.
  - ItemKNN giảm khoảng −0,003.
- **VSTAN:** paired bootstrap có ý nghĩa thống kê (+0,0013, CI [+0,0009; +0,0016]) nhưng mức tăng rất nhỏ so với khoảng cách giữa các mô hình.
- **Bài toán 2 (thêm giỏ/mua):**
  - Với `train_validation_refit`, phiên 60 phút tăng nhẹ ở VSTAN (+0,008), SR-GNN (+0,007) và GRU4Rec (+0,005).
  - Với `train_only`, GRU4Rec giảm (−0,009).
  - Chiều thay đổi không nhất quán giữa hai biến thể.
- **Thứ hạng mô hình không đổi:** Reranker > GRU4Rec > SR-GNN ≈ VSTAN/STAN > Repeat-last.
- **Cấu hình chọn trên validation 60 phút gần như trùng với 30 phút.** Chỉ hai lựa chọn khác:
  - VSTAN chọn idf 2.0 thay vì 0.5;
  - reranker chọn grid0 thay vì grid1.
  Số epoch GRU4Rec (2) và SR-GNN (5) giữ nguyên.

**Khuyến nghị: giữ ngưỡng phiên 30 phút làm giao thức chính.** Kéo dài lên 60 phút không cải thiện mô hình tốt nhất, và lợi ích ở các mô hình khác chỉ ở mức phần nghìn. Số liệu trên `q60` (95.783 query) dùng bộ query khác, nên chỉ để tham khảo, không dùng để so trực tiếp với `reports/final`.
